# Laboratorio 3: Codificación de fuente (Parte 2)
**Curso:** Teoría de la Información Transmisión de Datos

Esta actividad constituye la segunda parte del laboratorio de codificación de fuente. En la primera sesión, se implementó un flujo completo de compresión sin pérdida a partir del archivo “[Quijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1307721/mod_assign/introattachment/0/Quijote.txt?forcedownload=1)”, considerando cada carácter como símbolo, construyendo el código de Huffman, generando el fichero comprimido, decodificándolo y verificando su integridad, así como evaluando su eficiencia en términos de tamaño y entropía.

En esta segunda sesión, profundizaremos en el análisis y exploraremos variaciones metodológicas para comprender mejor el impacto del modelado de la fuente y de las propiedades estadísticas del texto sobre la eficiencia de compresión. Se propone trabajar en dos direcciones:

* Modelado por k-gramas y comparación de eficiencia para distintos valores de k.
* Repetición del flujo sobre un archivo con distribución estadística distinta ("Unijote.txt") para contrastar resultados.

## Ejercicio 1: Huffman por k-gramas y comparación (k = 1, 2, 3, 4)

Extender el flujo de compresión sin pérdida con Huffman para modelar la fuente mediante k-gramas (bloques de k caracteres) y comparar la eficiencia frente al caso k=1, incluyendo la comparación con un compresor estándar.

**Dataset**

Usar el archivo de texto plano "[Quijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1307721/mod_assign/introattachment/0/Quijote.txt?forcedownload=1)" (UTF-8).

**Tareas**

* Lectura del fichero. Leer "[Quijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1307721/mod_assign/introattachment/0/Quijote.txt?forcedownload=1)" preservando UTF-8.

* Modelado por k-gramas. Para cada $k ∈ {1, 2, 3, 4}$:

* Calcular frecuencias absoluta y relativa de los k-gramas.

* Reportar: tamaño en bytes/caracteres, número de símbolos distintos, entropía $H_k$ por token y entropía por carácter $H_k/k$.

* Construcción del código de Huffman. Generar el código prefijo óptimo para cada alfabeto de k-gramas.

* Codificación y decodificación. Producir y verificar ficheros .huff.k.

* Comparación de eficiencia.

* Medir tamaño de cabecera, tamaño total, bits/caracter y ratio de compresión para cada $k$.

* Comparar el resultado con el tamaño obtenido al comprimir el mismo archivo usando un compresor estándar (por ejemplo, zip), comentando ventajas y desventajas observadas.

* Analizar el efecto de aumentar $k$ en la eficiencia de compresión, la relación con la entropía teórica y el desempeño del compresor estándar.

In [19]:
import heapq
from collections import defaultdict
import math
import urllib.request
import os
from decimal import Decimal, getcontext
import time
from tabulate import tabulate
import zipfile
import io

# Establece la precisión deseada (50 dígitos)
getcontext().prec = 50

################################################################################################
# 1. Configuración inicial
################################################################################################
archivos_diccionario = {
    'Quijote.txt': '1IBWsnFZ3OsFrFosYOQFJ8BRulLEBs3Cq'
}
k_lista = [1, 2, 3, 4]

resultados = []
resultados_zip = []


################################################################################################
# 2. Funciones auxiliares
################################################################################################
def leer_archivo(archivo):
    """Lee archivo desde Google Drive"""
    try:
        url = f"https://drive.google.com/uc?export=download&id={archivos_diccionario[archivo]}"
        respuesta = urllib.request.urlopen(url)
        texto = respuesta.read().decode('utf-8')
        return texto
    except Exception as e:
        print(f"Error leyendo archivo {archivo}: {e}")
        return ""


def tabla_probabilidades(frecuencia, total_elementos):
    """Calcula probabilidades para cada símbolo"""
    if total_elementos <= 0:
        return frecuencia, Decimal(0)

    probabilidad_total = Decimal(0)
    for simbolo in frecuencia:
        ocurrencias = frecuencia[simbolo][0]
        probabilidad = Decimal(ocurrencias) / total_elementos
        frecuencia[simbolo][1] = probabilidad
        probabilidad_total += probabilidad

    return frecuencia, probabilidad_total


def calcular_entropia(frecuencia):
    """Calcula la entropía de la fuente"""
    H = Decimal(0)
    for datos in frecuencia.values():
        probabilidad = datos[1]
        if probabilidad > 0:
            try:
                log_prob = Decimal(math.log2(probabilidad))
                H -= probabilidad * log_prob
            except (ValueError, OverflowError):
                continue
    return H


def huffman_mejorado(frecuencias_dict):
    """Genera códigos Huffman"""
    if not frecuencias_dict:
        return []

    if len(frecuencias_dict) == 1:
        simbolo = list(frecuencias_dict.keys())[0]
        frecuencias_dict[simbolo][2] = '0'
        return [(simbolo, '0')]

    try:
        heap = []
        for simbolo, datos in frecuencias_dict.items():
            probabilidad = datos[1]
            heap.append([probabilidad, [simbolo, ""]])

        heapq.heapify(heap)

        while len(heap) > 1:
            lo = heapq.heappop(heap)
            hi = heapq.heappop(heap)

            for pair in lo[1:]:
                pair[1] = '0' + pair[1]
            for pair in hi[1:]:
                pair[1] = '1' + pair[1]

            nueva_probabilidad = lo[0] + hi[0]
            heapq.heappush(heap, [nueva_probabilidad] + lo[1:] + hi[1:])

        codigos = heapq.heappop(heap)[1:]
        codigos_ordenados = sorted(codigos, key=lambda p: (len(p[1]), p[0]))

        for simbolo, codigo in codigos_ordenados:
            frecuencias_dict[simbolo][2] = codigo

        return codigos_ordenados

    except Exception as e:
        print(f"Error en algoritmo Huffman: {e}")
        return []


def calcular_metricas(frecuencia, H, total_chars, codigos):
    """Calcula todas las métricas de compresión"""
    # Calcular L (longitud promedio)
    suma_ponderada = Decimal(0)
    acumulador_real = Decimal(0)

    for datos in frecuencia.values():
        probabilidad = datos[1]
        ocurrencia = datos[0]
        codigo = datos[2]

        longitud_codigo = len(codigo)
        valor_ponderado = probabilidad * longitud_codigo
        acumulador_real += ocurrencia * longitud_codigo

        suma_ponderada += valor_ponderado

    L = suma_ponderada
    L_real = acumulador_real / total_chars

    # Calcular eficiencia
    eff = H / L if L > 0 else Decimal(0)
    eff_real = H / L_real if L_real > 0 else Decimal(0)

    # Calcular tamaño del diccionario
    tamaño_diccionario = 0
    for simbolo, codigo in codigos:
        simb_utf8 = simbolo.encode('utf-8')
        bytes_codigo = (len(codigo) + 7) // 8
        tamaño_diccionario += 1 + len(simb_utf8) + 1 + bytes_codigo

    return L, L_real, eff, eff_real, tamaño_diccionario


def comprimir_archivo(texto, frecuencia, codigos, nombre_archivo, k):
    """Comprime el texto y guarda en archivo .huff - CORREGIDA"""
    try:
        # Crear mapping rápido para codificación
        codigo_map = {simbolo: codigo for simbolo, codigo in codigos}

        # print("Codificando texto...")
        # Codificar el texto COMPLETO en bloques de k caracteres
        cadena_de_bits = ''
        total_bloques = len(texto) - k + 1
        for i in range(total_bloques):
            bloque = texto[i:i + k]
            cadena_de_bits += codigo_map.get(bloque, '')

        # print(f"Texto codificado. Bits: {len(cadena_de_bits)}, Bloques: {total_bloques}")

        # Calcular padding
        tamanio_de_relleno = (8 - len(cadena_de_bits) % 8) % 8
        cadena_de_bits += '0' * tamanio_de_relleno

        # Convertir a bytes
        bits_bytes = bytearray()
        for i in range(0, len(cadena_de_bits), 8):
            byte_str = cadena_de_bits[i:i + 8]
            byte_val = int(byte_str, 2)
            bits_bytes.append(byte_val)

        # Calcular overhead del diccionario
        num_simbolos = len(codigos)
        # print(f"Número de símbolos: {num_simbolos}")

        # Usar 4 bytes para almacenar el número de símbolos si es necesario
        bytes_para_num_simbolos = 4 if num_simbolos > 65535 else 2
        tamanio_overhead = 1 + bytes_para_num_simbolos  # 1 byte padding + X bytes num símbolos

        # GUARDAR EL VALOR DE K Y LONGITUD ORIGINAL EN EL ARCHIVO
        tamanio_overhead += 1 + 4  # 1 byte para k + 4 bytes para longitud original

        for simbolo, codigo in codigos:
            simb_utf8 = simbolo.encode('utf-8')
            bytes_necesarios_codigo = (len(codigo) + 7) // 8
            tamanio_overhead += 1 + len(simb_utf8) + 1 + bytes_necesarios_codigo

        nombre_archivo_salida = f"{nombre_archivo}.huf.k{k}"
        # print(f"Guardando archivo comprimido: {nombre_archivo_salida}")

        with open(nombre_archivo_salida, "wb") as f:
            # Metadata - padding
            f.write(tamanio_de_relleno.to_bytes(1, 'big'))

            # Metadata - valor de k
            f.write(k.to_bytes(1, 'big'))

            # Metadata - longitud original del texto
            longitud_original = len(texto)
            f.write(longitud_original.to_bytes(4, 'big'))

            # Metadata - número de símbolos (2 o 4 bytes)
            if num_simbolos > 65535:
                f.write(b'\xFF\xFF')  # Marcador para indicar que usamos 4 bytes
                f.write(num_simbolos.to_bytes(4, 'big'))
            else:
                f.write(num_simbolos.to_bytes(2, 'big'))

            # Diccionario
            for simbolo, codigo in codigos:
                simb_utf8 = simbolo.encode('utf-8')
                f.write(len(simb_utf8).to_bytes(1, 'big'))
                f.write(simb_utf8)
                f.write(len(codigo).to_bytes(1, 'big'))

                codigo_int = int(codigo, 2)
                bytes_necesarios = (len(codigo) + 7) // 8
                f.write(codigo_int.to_bytes(bytes_necesarios, 'big'))

            # Datos comprimidos
            f.write(bits_bytes)

        tamaño_comprimido = os.path.getsize(nombre_archivo_salida)
        # print(f"Archivo guardado. Tamaño: {tamaño_comprimido} bytes")
        return nombre_archivo_salida, tamaño_comprimido, tamanio_overhead

    except Exception as e:
        print(f"Error en compresión: {e}")
        import traceback
        traceback.print_exc()
        return None, 0, 0


def construir_trie(mapa_decodificacion):
    """Construye un árbol binario para decodificación eficiente"""
    raiz = {}
    for codigo, simbolo in mapa_decodificacion.items():
        nodo = raiz
        for bit in codigo:
            nodo = nodo.setdefault(bit, {})
        nodo['simbolo'] = simbolo
    return raiz

def decodificar_con_trie(cadena_bits, trie):
    """Decodifica los bits usando el árbol (trie)"""
    nodo = trie
    bloques = []
    for bit in cadena_bits:
        if bit not in nodo:
            raise ValueError("Secuencia inválida en los bits")
        nodo = nodo[bit]
        if 'simbolo' in nodo:
            bloques.append(nodo['simbolo'])
            nodo = trie  # Reiniciar al inicio del trie
    return bloques

def descomprimir_y_validar(nombre_archivo_comprimido, texto_original):
    """Descomprime y valida integridad - OPTIMIZADO CON TRIE"""
    if not os.path.exists(nombre_archivo_comprimido):
        return False

    try:
        with open(nombre_archivo_comprimido, "rb") as f:
            # Leer metadata
            tamanio_de_relleno = int.from_bytes(f.read(1), 'big')
            k = int.from_bytes(f.read(1), 'big')
            longitud_original = int.from_bytes(f.read(4), 'big')

            # Leer número de símbolos
            first_two_bytes = f.read(2)
            if first_two_bytes == b'\xFF\xFF':
                num_simbolos = int.from_bytes(f.read(4), 'big')
            else:
                num_simbolos = int.from_bytes(first_two_bytes, 'big')

            print(f"Descomprimiendo {num_simbolos} símbolos para k={k}...")

            # Leer diccionario
            mapa_decodificacion = {}
            for _ in range(num_simbolos):
                len_simb = int.from_bytes(f.read(1), 'big')
                simbolo_bytes = f.read(len_simb)
                simbolo = simbolo_bytes.decode('utf-8')
                len_codigo = int.from_bytes(f.read(1), 'big')
                bytes_necesarios = (len_codigo + 7) // 8
                codigo_int = int.from_bytes(f.read(bytes_necesarios), 'big')
                codigo = bin(codigo_int)[2:].zfill(len_codigo)
                mapa_decodificacion[codigo] = simbolo

            datos_comprimidos = f.read()

        # Convertir bytes a bits (string de bits)
        cadena_de_bits = ''.join(bin(byte)[2:].zfill(8) for byte in datos_comprimidos)
        if tamanio_de_relleno > 0:
            cadena_de_bits = cadena_de_bits[:-tamanio_de_relleno]

        # print(f"Bits a decodificar: {len(cadena_de_bits)}")

        # ⏱️ Decodificación optimizada usando Trie
        trie = construir_trie(mapa_decodificacion)
        bloques_decodificados = decodificar_con_trie(cadena_de_bits, trie)

        # print(f"Bloques decodificados: {len(bloques_decodificados)}")

        # RECONSTRUIR TEXTO
        if k == 1:
            texto_decodificado = ''.join(bloques_decodificados)
        else:
            if not bloques_decodificados:
                texto_decodificado = ""
            else:
                texto_decodificado = bloques_decodificados[0]
                for i in range(1, len(bloques_decodificados)):
                    bloque = bloques_decodificados[i]
                    if len(bloque) == k:
                        texto_decodificado += bloque[-1]
                    else:
                        texto_decodificado += bloque

        # print(f"Texto decodificado: {len(texto_decodificado)} caracteres")
        # print(f"Texto original: {len(texto_original)} caracteres")
        # print(f"Longitud esperada: {longitud_original} caracteres")

        # Validación de longitud
        if len(texto_decodificado) != longitud_original:
            print(f"ERROR: Longitud incorrecta. Esperado: {longitud_original}, Obtenido: {len(texto_decodificado)}")
            print(f"Original (inicio): {texto_original[:100]}")
            print(f"Decodificado (inicio): {texto_decodificado[:100]}")
            return False

        # Validación de contenido
        if texto_decodificado != texto_original:
            print("ERROR: Contenido diferente")
            for i in range(min(len(texto_original), len(texto_decodificado))):
                if texto_original[i] != texto_decodificado[i]:
                    print(f"Primera diferencia en posición {i}:")
                    print(f"Original: '{texto_original[i]}' (ASCII: {ord(texto_original[i])})")
                    print(f"Decodificado: '{texto_decodificado[i]}' (ASCII: {ord(texto_decodificado[i])})")
                    start = max(0, i - 10)
                    end = min(len(texto_original), i + 11)
                    print(f"Contexto original: '{texto_original[start:end]}'")
                    print(f"Contexto decodificado: '{texto_decodificado[start:end]}'")
                    break
            return False

        return True

    except Exception as e:
        print(f"Error en descompresión: {e}")
        import traceback
        traceback.print_exc()
        return False


def comprimir_con_zip(texto, nombre_archivo):
    """Comprime el texto usando ZIP estándar"""
    nombre_zip = f"{nombre_archivo}.zip"

    # Crear archivo ZIP en memoria
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.writestr(nombre_archivo, texto)

    # Guardar archivo ZIP
    with open(nombre_zip, 'wb') as f:
        f.write(zip_buffer.getvalue())

    tamaño_zip = os.path.getsize(nombre_zip)
    return nombre_zip, tamaño_zip


################################################################################################
# 3. Procesamiento principal
################################################################################################
def procesar_archivo_completo(nombre_archivo, k):
    """Procesa un archivo completo con un valor k específico"""
    print(f"\n{'=' * 80}")
    print(f"PROCESANDO: {nombre_archivo} con k={k}")
    print(f"{'=' * 80}")

    start_time = time.time()

    try:
        # 1. Leer archivo
        texto = leer_archivo(nombre_archivo)
        if not texto:
            print(f"✗ Error: No se pudo leer el archivo {nombre_archivo}")
            return None

        texto_original = texto
        total_chars = len(texto)

        if k > len(texto):
            print(f"✗ Error: k={k} es mayor que la longitud del texto ({len(texto)})")
            return None

        # 2. Calcular tabla de frecuencias
        frecuencia = defaultdict(lambda: [0, Decimal(0), ''])
        for i in range(0, len(texto) - k + 1):
            bloque = texto[i:i + k]
            frecuencia[bloque][0] += 1

        total_bloques = len(texto) - k + 1

        # 3. Calcular probabilidades
        frecuencia, prob_total = tabla_probabilidades(frecuencia, total_bloques)

        # 4. Calcular entropía
        H = calcular_entropia(frecuencia)
        H_por_caracter = H / k


        # 5. Generar códigos Huffman
        codigos = huffman_mejorado(frecuencia)
        if not codigos:
            print("✗ Error: No se pudieron generar códigos Huffman")
            return None

        # 6. Calcular métricas
        L, L_real, eff, eff_real, tamaño_diccionario = calcular_metricas(
            frecuencia, H, total_bloques, codigos
        )

        # 7. Comprimir archivo - usar el texto COMPLETO, no la muestra
        nombre_base = nombre_archivo.split('.')[0]
        nombre_comprimido, tamaño_comprimido, overhead = comprimir_archivo(
            texto, frecuencia, codigos, nombre_base, k  # ← Usar 'texto' no 'texto_muestra'
        )
        if not nombre_comprimido:
            return None

        # 8. Validar descompresión
        valido = descomprimir_y_validar(nombre_comprimido, texto)  # ← Pasar texto completo

        # 9. Calcular relación de compresión
        tamaño_original = len(texto_original.encode('utf-8'))
        relacion_compresion = Decimal(tamaño_original) / tamaño_comprimido if tamaño_comprimido > 0 else Decimal(0)

        # 10. Guardar resultados
        resultado = {
        'archivo': nombre_archivo,
        'k': k,
        'simbolos': len(frecuencia),
        'entropia': float(H),
        'entropia_por_caracter': float(H_por_caracter),  # ✅ <--- AGREGAR ESTA LÍNEA
        'L_cx': float(L),
        'eficiencia': float(eff * 100),
        'tamaño_diccionario': tamaño_diccionario,
        'tamaño_comprimido': tamaño_comprimido,
        'overhead': overhead,
        'relacion_compresion': float(relacion_compresion),
        'valido': valido,
        'tiempo_procesamiento': time.time() - start_time,
        'tamaño_original': tamaño_original
        }

        # 11. Mostrar resultados con formato mejorado
        tabla_detalle = [
            ["Símbolos únicos", f"{len(frecuencia):,}"],
            ["Entropía H(X)", f"{H:.6f} bits/símbolo"],
            ["Entropía por carácter H(X)/k", f"{H_por_caracter:.6f} bits/carácter"],
            ["Longitud promedio L(C,X)", f"{L:.6f} bits/símbolo"],
            ["Eficiencia", f"{eff * 100:.2f}%"],
            ["Tamaño diccionario", f"{tamaño_diccionario:,} bytes"],
            ["Tamaño original", f"{tamaño_original:,} bytes"],
            ["Tamaño comprimido", f"{tamaño_comprimido:,} bytes"],
            ["Overhead", f"{overhead:,} bytes"],
            ["Relación compresión", f"{relacion_compresion:.2f}x"],
            ["Validación", "✓ ÉXITO" if valido else "✗ FALLO"],
            ["Tiempo procesamiento", f"{time.time() - start_time:.2f} segundos"]
        ]

        print(tabulate(tabla_detalle, tablefmt="grid"))

        return resultado

    except Exception as e:
        print(f"✗ Error procesando {nombre_archivo} con k={k}: {e}")
        import traceback
        traceback.print_exc()
        return None


def procesar_zip_archivos():
    """Procesa compresión ZIP para todos los archivos"""
    global resultados_zip

    print(f"\n{'=' * 80}")
    print("PROCESANDO COMPRESIÓN ZIP")
    print(f"{'=' * 80}")

    for nombre_archivo in archivos_diccionario.keys():
        start_time = time.time()

        try:
            # Leer archivo
            texto = leer_archivo(nombre_archivo)
            if not texto:
                continue

            tamaño_original = len(texto.encode('utf-8'))

            # Comprimir con ZIP
            nombre_base = nombre_archivo.split('.')[0]
            nombre_zip, tamaño_zip = comprimir_con_zip(texto, nombre_archivo)

            # Calcular relación de compresión
            relacion_compresion = Decimal(tamaño_original) / tamaño_zip if tamaño_zip > 0 else Decimal(0)

            resultado_zip = {
                'archivo': nombre_archivo,
                'metodo': 'ZIP',
                'tamaño_original': tamaño_original,
                'tamaño_comprimido': tamaño_zip,
                'relacion_compresion': float(relacion_compresion),
                'tiempo_procesamiento': time.time() - start_time
            }

            resultados_zip.append(resultado_zip)

            tabla_detalle_zip = [
                ["Método", "ZIP"],
                ["Archivo", nombre_archivo],
                ["Tamaño original", f"{tamaño_original:,} bytes"],
                ["Tamaño comprimido", f"{tamaño_zip:,} bytes"],
                ["Relación compresión", f"{relacion_compresion:.2f}x"],
                ["Tiempo procesamiento", f"{time.time() - start_time:.2f} segundos"]
            ]

            print(tabulate(tabla_detalle_zip, tablefmt="grid"))
            print()

        except Exception as e:
            print(f"✗ Error procesando ZIP para {nombre_archivo}: {e}")


################################################################################################
# 4. Ejecución principal
################################################################################################
def main():
    """Función principal que procesa todos los archivos y valores de k"""
    global resultados, resultados_zip

    print("INICIANDO PROCESAMIENTO DE ARCHIVOS...")
    print(f"Archivos a procesar: {list(archivos_diccionario.keys())}")
    print(f"Valores de k: {k_lista}")
    print()

    # Procesar compresión ZIP primero
    procesar_zip_archivos()

    # Procesar compresión Huffman para todos los k
    for nombre_archivo in archivos_diccionario.keys():
        for k in k_lista:
            resultado = procesar_archivo_completo(nombre_archivo, k)
            if resultado:
                resultados.append(resultado)

    # Mostrar resumen final con tabulate
    if resultados:
        print(f"\n{'#' * 120}")
        print("RESUMEN FINAL DE RESULTADOS HUFFMAN")
        print(f"{'#' * 120}")

        headers = ['Archivo', 'K', 'Símbolos', 'Entropía/token', 'Entropía/carácter', 'L(C,X)', 'Eficiencia%',
                   'Diccionario', 'Comprimido', 'Relación', 'Válido', 'Tiempo(s)']




        tabla_resumen = []
        for res in resultados:
            tabla_resumen.append([
                res['archivo'],
                res['k'],
                f"{res['simbolos']:,}",
                f"{res['entropia']:.4f}",            # entropía por token
                f"{res['entropia_por_caracter']:.4f}",  # entropía por carácter
                f"{res['L_cx']:.4f}",
                f"{res['eficiencia']:.2f}",
                f"{res['tamaño_diccionario']:,}",
                f"{res['tamaño_comprimido']:,}",
                f"{res['relacion_compresion']:.2f}x",
                '✓' if res['valido'] else '✗',
                f"{res['tiempo_procesamiento']:.2f}"
            ])

        # Imprimir tabla con formato elegante
        print(tabulate(tabla_resumen, headers=headers, tablefmt="grid", numalign="right", stralign="center"))

        # Mostrar comparación con ZIP
        print(f"\n{'#' * 120}")
        print("COMPARACIÓN HUFFMAN vs ZIP")
        print(f"{'#' * 120}")

        comparacion_headers = ['Archivo', 'Método', 'K', 'Tamaño', 'Relación', 'Tiempo(s)']
        comparacion_tabla = []

        # Agregar resultados ZIP
        for zip_res in resultados_zip:
            comparacion_tabla.append([
                zip_res['archivo'],
                'ZIP',
                '-',
                f"{zip_res['tamaño_comprimido']:,}",
                f"{zip_res['relacion_compresion']:.2f}x",
                f"{zip_res['tiempo_procesamiento']:.2f}"
            ])

        # Agregar resultados Huffman
        for huff_res in resultados:
            comparacion_tabla.append([
                huff_res['archivo'],
                'HUFFMAN',
                huff_res['k'],
                f"{huff_res['tamaño_comprimido']:,}",
                f"{huff_res['relacion_compresion']:.2f}x",
                f"{huff_res['tiempo_procesamiento']:.2f}"
            ])

        print(tabulate(comparacion_tabla, headers=comparacion_headers, tablefmt="grid", numalign="right",
                       stralign="center"))

        # Encontrar la mejor compresión por archivo
        print(f"\n{'#' * 120}")
        print("MEJOR MÉTODO DE COMPRESIÓN POR ARCHIVO")
        print(f"{'#' * 120}")

        mejores_headers = ['Archivo', 'Mejor Método', 'Mejor K', 'Mejor Relación', 'Tamaño']
        mejores_tabla = []

        for archivo in archivos_diccionario.keys():
            # Encontrar mejor resultado Huffman para este archivo
            huffman_results = [r for r in resultados if r['archivo'] == archivo]
            best_huffman = max(huffman_results, key=lambda x: x['relacion_compresion']) if huffman_results else None

            # Encontrar resultado ZIP para este archivo
            zip_result = next((r for r in resultados_zip if r['archivo'] == archivo), None)

            if best_huffman and zip_result:
                if best_huffman['relacion_compresion'] > zip_result['relacion_compresion']:
                    mejores_tabla.append([
                        archivo,
                        'HUFFMAN',
                        best_huffman['k'],
                        f"{best_huffman['relacion_compresion']:.2f}x",
                        f"{best_huffman['tamaño_comprimido']:,}"
                    ])
                else:
                    mejores_tabla.append([
                        archivo,
                        'ZIP',
                        '-',
                        f"{zip_result['relacion_compresion']:.2f}x",
                        f"{zip_result['tamaño_comprimido']:,}"
                    ])

        print(tabulate(mejores_tabla, headers=mejores_headers, tablefmt="grid", numalign="right", stralign="center"))

        # Guardar resultados en CSV
        with open('resultados_compresion.csv', 'w', encoding='utf-8') as f:
            # Escribir headers
            f.write(
                "Tipo;Archivo;K;Simbolos;Entropia;L_CX;Eficiencia%;Tamaño_Diccionario;Tamaño_Comprimido;Relacion_Compresion;Valido;Tiempo\n")

            # Escribir resultados Huffman
            for res in resultados:
                f.write(f"HUFFMAN;{res['archivo']};{res['k']};{res['simbolos']};{res['entropia']:.6f};")
                f.write(f"{res['L_cx']:.6f};{res['eficiencia']:.6f};{res['tamaño_diccionario']};")
                f.write(f"{res['tamaño_comprimido']};{res['relacion_compresion']:.6f};")
                f.write(f"{'SI' if res['valido'] else 'NO'};{res['tiempo_procesamiento']:.2f}\n")

            # Escribir resultados ZIP
            for res in resultados_zip:
                f.write(
                    f"ZIP;{res['archivo']};;;;;;;{res['tamaño_comprimido']};{res['relacion_compresion']:.6f};;{res['tiempo_procesamiento']:.2f}\n")

        print(f"\n✓ Resultados guardados en 'resultados_compresion.csv'")
        print(f"✓ Total de procesamientos Huffman: {len(resultados)}")
        print(f"✓ Total de procesamientos ZIP: {len(resultados_zip)}")
        print(f"✓ Procesamientos exitosos: {sum(1 for r in resultados if r['valido'])}")
    else:
        print("✗ No se completó ningún procesamiento exitoso")


if __name__ == "__main__":
    main()

INICIANDO PROCESAMIENTO DE ARCHIVOS...
Archivos a procesar: ['Quijote.txt']
Valores de k: [1, 2, 3, 4]


PROCESANDO COMPRESIÓN ZIP
+----------------------+-----------------+
| Método               | ZIP             |
+----------------------+-----------------+
| Archivo              | Quijote.txt     |
+----------------------+-----------------+
| Tamaño original      | 2,141,519 bytes |
+----------------------+-----------------+
| Tamaño comprimido    | 799,729 bytes   |
+----------------------+-----------------+
| Relación compresión  | 2.68x           |
+----------------------+-----------------+
| Tiempo procesamiento | 1.36 segundos   |
+----------------------+-----------------+


PROCESANDO: Quijote.txt con k=1
Descomprimiendo 91 símbolos para k=1...
+------------------------------+------------------------+
| Símbolos únicos              | 91                     |
+------------------------------+------------------------+
| Entropía H(X)                | 4.378741 bits/símbolo  |
+---

**Discuta aquí la metodología, decisiones de formato y resultados obenidos**



## **Proceso de Compresión sobre *Quijote.txt***

### **Metodología Empleada**

Se evaluo el desempeño del algoritmo de **compresión de Huffman** adaptado a diferentes valores de longitud de bloque *k* (con *k = 1, 2, 3, 4*), comparando los resultados obtenidos con los de una compresión estándar mediante el formato **ZIP**

El proceso incluyó:

* Lectura del archivo completo
* Segmentación del texto en bloques de longitud *k*
* Construcción del diccionario Huffman con base en la frecuencia de dichos bloques
* Codificación del texto
* Medición de métricas como entropía, longitud promedio del código, eficiencia, tamaño del archivo comprimido, y tiempo de ejecución.
* Validación del proceso mediante descompresión y comparación binaria con el original.


### **Decisiones de Formato**

* El archivo comprimido incluye el **diccionario Huffman (como cabecera)** (overhead), lo que afecta el tamaño total del archivo comprimido, especialmente para valores altos de *k* con muchos símbolos únicos
* El procesamiento fue realizado sobre el texto completo del archivo *Quijote.txt* (\~2 MB), lo cual provee un conjunto de datos suficientemente amplio para analizar el comportamiento de la compresión bajo distintos contextos


### **Resultados Obtenidos**

#### **ZIP (Referencia Estándar)**

* **Tamaño comprimido**: 799,729 bytes
* **Relación de compresión**: **2.68x**
* **Tiempo de procesamiento**: 1.65 s
* Método más eficiente en este caso en términos de relación de compresión y velocidad

#### **Compresión Huffman para distintos *k***

| **k** | Símbolos únicos | H(X)/k (bits/caracter) | L(C,X)/k | Eficiencia | Tamaño comprimido | Relación | Tiempo (s) |
| ----: | --------------: | ---------------------: | -------: | ---------: | ----------------: | -------: | ---------: |
|     1 |              91 |                   4.38 |     4.42 |     99.16% |   1,158,476 bytes |    1.85x |       3.89 |
|     2 |           1,671 |                   3.85 |     3.86 |     99.65% |   2,035,415 bytes |    1.05x |       5.75 |
|     3 |          11,642 |                   3.48 |     3.49 |     99.73% |   2,835,056 bytes |    0.76x |       8.35 |
|     4 |          49,671 |                   3.16 |     3.17 |     99.78% |   3,775,357 bytes |    0.57x |      12.93 |

#### **Mejor resultado Huffman:**

* Para *k = 1*: relación de compresión **1.85x**

---

### **Comparación Huffman vs ZIP**

| Criterio               | Huffman (*k=1*) | ZIP           |
| ---------------------- | --------------- | ------------- |
| Relación compresión    | 1.85x           | **2.68x**     |
| Tamaño comprimido      | 1,158,476 B     | **799,729 B** |
| Tiempo procesamiento   | 3.89 s          | **1.65 s**    |
| Compresión sin pérdida | ✅               | ✅             |

#### **Ventajas de ZIP:**

* Mayor compresión
* Más rápido

#### **Ventajas de Huffman (k=1):**

* Codificación óptima estadísticamente
* Alta eficiencia (>99%) en representación binaria de símbolos
* Proceso controlado


### **Análisis del Aumento de *k***

#### **Relación con la Entropía:**

* A medida que *k* aumenta:

  * La entropía por símbolo H(X) **aumenta**, porque los bloques son más largos (hay mas informacion por cada bloque)
  * La entropía por carácter (H(X)/k) **disminuye**: es más eficiente modelar la fuente a nivel de bloques.

  Ejemplo:

  * *k=1*: H(X)/k ≈ 4.38 bits/carácter
  * *k=4*: H(X)/k ≈ 3.16 bits/carácter

#### **Sin embargo**, esto ***no se traduce en mejor compresión***:

* El número de símbolos únicos crece con *k*
* Aumenta el tamaño del diccionario (*overhead*):

  * 460 bytes para *k=1* vs 450,268 bytes para *k=4*.
* Esto **contrarresta las ganancias teóricas** por la mayor entropia
* A mayor *k*, tambien se ve **más tiempo de procesamiento**, **mayor tamaño final**, y **menor relación de compresión**

#### **Resultado Final:**

* Aunque la eficiencia de codificación mejora ligeramente con *k*, la **relación de compresión empeora drásticamente** por el aumento de tamaño del diccionario.


## **Conclusiones**

* **ZIP** es el método más efectivo y práctico para comprimir *Quijote.txt*, logrando **la mejor relación de compresión y menor tiempo**
* **Huffman con *k=1*** es el mejor entre los métodos evaluados de Huffman, aunque no supera a ZIP
* **Aumentar *k* mejora la modelación de la fuente (menor entropía por carácter), pero empeora la compresión final** debido al crecimiento del diccionario

## Ejercicio 2: Repetición del flujo con otro fichero

Repetir el flujo de compresión sin pérdida sobre un segundo archivo y comparar los resultados con "Quijote.txt", evaluando la compresibilidad de un texto con distribución estadística distinta, e incluyendo comparación con un compresor estándar.

**Dataset**

Usar el archivo de texto plano "[Unijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1307721/mod_assign/introattachment/0/Unijote.txt?forcedownload=1)" (UTF-8).

**Tareas**

* Lectura del fichero y modelado. Calcular frecuencias y entropía (k=1).

* Construcción del código de Huffman y codificación. Generar fichero .huff.

* Decodificación y verificación. Comprobar igualdad exacta con el original.

* Comparación y análisis.

* Comparar tamaño original y comprimido con los de "Quijote.txt".

* Comprimir el archivo con un compresor estándar (por ejemplo, zip) y comparar tamaños, discutiendo si Huffman resulta competitivo en este escenario.

**Opcional**:

Repetir para k=2 y k=3 y comparar tendencias. (*Punto adicional en la eveluacion del laboratorio*)

In [18]:
import heapq
from collections import defaultdict
import math
import urllib.request
import os
from decimal import Decimal, getcontext
import time
from tabulate import tabulate
import zipfile
import io

# Establece la precisión deseada (50 dígitos)
getcontext().prec = 50

################################################################################################
# 1. Configuración inicial
################################################################################################
archivos_diccionario = {
    'Quijote.txt': '1IBWsnFZ3OsFrFosYOQFJ8BRulLEBs3Cq',
    'Unijote.txt': '12I_pEn81rN8lq9yS3iuNeWGvXJ38gwRJ'
}
k_lista = [1, 2, 3, 4]

resultados = []
resultados_zip = []


################################################################################################
# 2. Funciones auxiliares
################################################################################################
def leer_archivo(archivo):
    """Lee archivo desde Google Drive"""
    try:
        url = f"https://drive.google.com/uc?export=download&id={archivos_diccionario[archivo]}"
        respuesta = urllib.request.urlopen(url)
        texto = respuesta.read().decode('utf-8')
        return texto
    except Exception as e:
        print(f"Error leyendo archivo {archivo}: {e}")
        return ""


def tabla_probabilidades(frecuencia, total_elementos):
    """Calcula probabilidades para cada símbolo"""
    if total_elementos <= 0:
        return frecuencia, Decimal(0)

    probabilidad_total = Decimal(0)
    for simbolo in frecuencia:
        ocurrencias = frecuencia[simbolo][0]
        probabilidad = Decimal(ocurrencias) / total_elementos
        frecuencia[simbolo][1] = probabilidad
        probabilidad_total += probabilidad

    return frecuencia, probabilidad_total


def calcular_entropia(frecuencia):
    """Calcula la entropía de la fuente"""
    H = Decimal(0)
    for datos in frecuencia.values():
        probabilidad = datos[1]
        if probabilidad > 0:
            try:
                log_prob = Decimal(math.log2(probabilidad))
                H -= probabilidad * log_prob
            except (ValueError, OverflowError):
                continue
    return H


def huffman_mejorado(frecuencias_dict):
    """Genera códigos Huffman"""
    if not frecuencias_dict:
        return []

    if len(frecuencias_dict) == 1:
        simbolo = list(frecuencias_dict.keys())[0]
        frecuencias_dict[simbolo][2] = '0'
        return [(simbolo, '0')]

    try:
        heap = []
        for simbolo, datos in frecuencias_dict.items():
            probabilidad = datos[1]
            heap.append([probabilidad, [simbolo, ""]])

        heapq.heapify(heap)

        while len(heap) > 1:
            lo = heapq.heappop(heap)
            hi = heapq.heappop(heap)

            for pair in lo[1:]:
                pair[1] = '0' + pair[1]
            for pair in hi[1:]:
                pair[1] = '1' + pair[1]

            nueva_probabilidad = lo[0] + hi[0]
            heapq.heappush(heap, [nueva_probabilidad] + lo[1:] + hi[1:])

        codigos = heapq.heappop(heap)[1:]
        codigos_ordenados = sorted(codigos, key=lambda p: (len(p[1]), p[0]))

        for simbolo, codigo in codigos_ordenados:
            frecuencias_dict[simbolo][2] = codigo

        return codigos_ordenados

    except Exception as e:
        print(f"Error en algoritmo Huffman: {e}")
        return []


def calcular_metricas(frecuencia, H, total_chars, codigos):
    """Calcula todas las métricas de compresión"""
    # Calcular L (longitud promedio)
    suma_ponderada = Decimal(0)
    acumulador_real = Decimal(0)

    for datos in frecuencia.values():
        probabilidad = datos[1]
        ocurrencia = datos[0]
        codigo = datos[2]

        longitud_codigo = len(codigo)
        valor_ponderado = probabilidad * longitud_codigo
        acumulador_real += ocurrencia * longitud_codigo

        suma_ponderada += valor_ponderado

    L = suma_ponderada
    L_real = acumulador_real / total_chars

    # Calcular eficiencia
    eff = H / L if L > 0 else Decimal(0)
    eff_real = H / L_real if L_real > 0 else Decimal(0)

    # Calcular tamaño del diccionario
    tamaño_diccionario = 0
    for simbolo, codigo in codigos:
        simb_utf8 = simbolo.encode('utf-8')
        bytes_codigo = (len(codigo) + 7) // 8
        tamaño_diccionario += 1 + len(simb_utf8) + 1 + bytes_codigo

    return L, L_real, eff, eff_real, tamaño_diccionario


def comprimir_archivo(texto, frecuencia, codigos, nombre_archivo, k):
    """Comprime el texto y guarda en archivo .huff - CORREGIDA"""
    try:
        # Crear mapping rápido para codificación
        codigo_map = {simbolo: codigo for simbolo, codigo in codigos}

        # print("Codificando texto...")
        # Codificar el texto COMPLETO en bloques de k caracteres
        cadena_de_bits = ''
        total_bloques = len(texto) - k + 1
        for i in range(total_bloques):
            bloque = texto[i:i + k]
            cadena_de_bits += codigo_map.get(bloque, '')

        # print(f"Texto codificado. Bits: {len(cadena_de_bits)}, Bloques: {total_bloques}")

        # Calcular padding
        tamanio_de_relleno = (8 - len(cadena_de_bits) % 8) % 8
        cadena_de_bits += '0' * tamanio_de_relleno

        # Convertir a bytes
        bits_bytes = bytearray()
        for i in range(0, len(cadena_de_bits), 8):
            byte_str = cadena_de_bits[i:i + 8]
            byte_val = int(byte_str, 2)
            bits_bytes.append(byte_val)

        # Calcular overhead del diccionario
        num_simbolos = len(codigos)
        # print(f"Número de símbolos: {num_simbolos}")

        # Usar 4 bytes para almacenar el número de símbolos si es necesario
        bytes_para_num_simbolos = 4 if num_simbolos > 65535 else 2
        tamanio_overhead = 1 + bytes_para_num_simbolos  # 1 byte padding + X bytes num símbolos

        # GUARDAR EL VALOR DE K Y LONGITUD ORIGINAL EN EL ARCHIVO
        tamanio_overhead += 1 + 4  # 1 byte para k + 4 bytes para longitud original

        for simbolo, codigo in codigos:
            simb_utf8 = simbolo.encode('utf-8')
            bytes_necesarios_codigo = (len(codigo) + 7) // 8
            tamanio_overhead += 1 + len(simb_utf8) + 1 + bytes_necesarios_codigo

        nombre_archivo_salida = f"{nombre_archivo}.huf.k{k}"
        # print(f"Guardando archivo comprimido: {nombre_archivo_salida}")

        with open(nombre_archivo_salida, "wb") as f:
            # Metadata - padding
            f.write(tamanio_de_relleno.to_bytes(1, 'big'))

            # Metadata - valor de k
            f.write(k.to_bytes(1, 'big'))

            # Metadata - longitud original del texto
            longitud_original = len(texto)
            f.write(longitud_original.to_bytes(4, 'big'))

            # Metadata - número de símbolos (2 o 4 bytes)
            if num_simbolos > 65535:
                f.write(b'\xFF\xFF')  # Marcador para indicar que usamos 4 bytes
                f.write(num_simbolos.to_bytes(4, 'big'))
            else:
                f.write(num_simbolos.to_bytes(2, 'big'))

            # Diccionario
            for simbolo, codigo in codigos:
                simb_utf8 = simbolo.encode('utf-8')
                f.write(len(simb_utf8).to_bytes(1, 'big'))
                f.write(simb_utf8)
                f.write(len(codigo).to_bytes(1, 'big'))

                codigo_int = int(codigo, 2)
                bytes_necesarios = (len(codigo) + 7) // 8
                f.write(codigo_int.to_bytes(bytes_necesarios, 'big'))

            # Datos comprimidos
            f.write(bits_bytes)

        tamaño_comprimido = os.path.getsize(nombre_archivo_salida)
        # print(f"Archivo guardado. Tamaño: {tamaño_comprimido} bytes")
        return nombre_archivo_salida, tamaño_comprimido, tamanio_overhead

    except Exception as e:
        print(f"Error en compresión: {e}")
        import traceback
        traceback.print_exc()
        return None, 0, 0


def construir_trie(mapa_decodificacion):
    """Construye un árbol binario para decodificación eficiente"""
    raiz = {}
    for codigo, simbolo in mapa_decodificacion.items():
        nodo = raiz
        for bit in codigo:
            nodo = nodo.setdefault(bit, {})
        nodo['simbolo'] = simbolo
    return raiz

def decodificar_con_trie(cadena_bits, trie):
    """Decodifica los bits usando el árbol (trie)"""
    nodo = trie
    bloques = []
    for bit in cadena_bits:
        if bit not in nodo:
            raise ValueError("Secuencia inválida en los bits")
        nodo = nodo[bit]
        if 'simbolo' in nodo:
            bloques.append(nodo['simbolo'])
            nodo = trie  # Reiniciar al inicio del trie
    return bloques

def descomprimir_y_validar(nombre_archivo_comprimido, texto_original):
    """Descomprime y valida integridad - OPTIMIZADO CON TRIE"""
    if not os.path.exists(nombre_archivo_comprimido):
        return False

    try:
        with open(nombre_archivo_comprimido, "rb") as f:
            # Leer metadata
            tamanio_de_relleno = int.from_bytes(f.read(1), 'big')
            k = int.from_bytes(f.read(1), 'big')
            longitud_original = int.from_bytes(f.read(4), 'big')

            # Leer número de símbolos
            first_two_bytes = f.read(2)
            if first_two_bytes == b'\xFF\xFF':
                num_simbolos = int.from_bytes(f.read(4), 'big')
            else:
                num_simbolos = int.from_bytes(first_two_bytes, 'big')

            print(f"Descomprimiendo {num_simbolos} símbolos para k={k}...")

            # Leer diccionario
            mapa_decodificacion = {}
            for _ in range(num_simbolos):
                len_simb = int.from_bytes(f.read(1), 'big')
                simbolo_bytes = f.read(len_simb)
                simbolo = simbolo_bytes.decode('utf-8')
                len_codigo = int.from_bytes(f.read(1), 'big')
                bytes_necesarios = (len_codigo + 7) // 8
                codigo_int = int.from_bytes(f.read(bytes_necesarios), 'big')
                codigo = bin(codigo_int)[2:].zfill(len_codigo)
                mapa_decodificacion[codigo] = simbolo

            datos_comprimidos = f.read()

        # Convertir bytes a bits (string de bits)
        cadena_de_bits = ''.join(bin(byte)[2:].zfill(8) for byte in datos_comprimidos)
        if tamanio_de_relleno > 0:
            cadena_de_bits = cadena_de_bits[:-tamanio_de_relleno]

        # print(f"Bits a decodificar: {len(cadena_de_bits)}")

        # ⏱️ Decodificación optimizada usando Trie
        trie = construir_trie(mapa_decodificacion)
        bloques_decodificados = decodificar_con_trie(cadena_de_bits, trie)

        # print(f"Bloques decodificados: {len(bloques_decodificados)}")

        # RECONSTRUIR TEXTO
        if k == 1:
            texto_decodificado = ''.join(bloques_decodificados)
        else:
            if not bloques_decodificados:
                texto_decodificado = ""
            else:
                texto_decodificado = bloques_decodificados[0]
                for i in range(1, len(bloques_decodificados)):
                    bloque = bloques_decodificados[i]
                    if len(bloque) == k:
                        texto_decodificado += bloque[-1]
                    else:
                        texto_decodificado += bloque

        # print(f"Texto decodificado: {len(texto_decodificado)} caracteres")
        # print(f"Texto original: {len(texto_original)} caracteres")
        # print(f"Longitud esperada: {longitud_original} caracteres")

        # Validación de longitud
        if len(texto_decodificado) != longitud_original:
            print(f"ERROR: Longitud incorrecta. Esperado: {longitud_original}, Obtenido: {len(texto_decodificado)}")
            print(f"Original (inicio): {texto_original[:100]}")
            print(f"Decodificado (inicio): {texto_decodificado[:100]}")
            return False

        # Validación de contenido
        if texto_decodificado != texto_original:
            print("ERROR: Contenido diferente")
            for i in range(min(len(texto_original), len(texto_decodificado))):
                if texto_original[i] != texto_decodificado[i]:
                    print(f"Primera diferencia en posición {i}:")
                    print(f"Original: '{texto_original[i]}' (ASCII: {ord(texto_original[i])})")
                    print(f"Decodificado: '{texto_decodificado[i]}' (ASCII: {ord(texto_decodificado[i])})")
                    start = max(0, i - 10)
                    end = min(len(texto_original), i + 11)
                    print(f"Contexto original: '{texto_original[start:end]}'")
                    print(f"Contexto decodificado: '{texto_decodificado[start:end]}'")
                    break
            return False

        return True

    except Exception as e:
        print(f"Error en descompresión: {e}")
        import traceback
        traceback.print_exc()
        return False


def comprimir_con_zip(texto, nombre_archivo):
    """Comprime el texto usando ZIP estándar"""
    nombre_zip = f"{nombre_archivo}.zip"

    # Crear archivo ZIP en memoria
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.writestr(nombre_archivo, texto)

    # Guardar archivo ZIP
    with open(nombre_zip, 'wb') as f:
        f.write(zip_buffer.getvalue())

    tamaño_zip = os.path.getsize(nombre_zip)
    return nombre_zip, tamaño_zip


################################################################################################
# 3. Procesamiento principal
################################################################################################
def procesar_archivo_completo(nombre_archivo, k):
    """Procesa un archivo completo con un valor k específico"""
    print(f"\n{'=' * 80}")
    print(f"PROCESANDO: {nombre_archivo} con k={k}")
    print(f"{'=' * 80}")

    start_time = time.time()

    try:
        # 1. Leer archivo
        texto = leer_archivo(nombre_archivo)
        if not texto:
            print(f"✗ Error: No se pudo leer el archivo {nombre_archivo}")
            return None

        texto_original = texto
        total_chars = len(texto)

        if k > len(texto):
            print(f"✗ Error: k={k} es mayor que la longitud del texto ({len(texto)})")
            return None

        # 2. Calcular tabla de frecuencias
        frecuencia = defaultdict(lambda: [0, Decimal(0), ''])
        for i in range(0, len(texto) - k + 1):
            bloque = texto[i:i + k]
            frecuencia[bloque][0] += 1

        total_bloques = len(texto) - k + 1

        # 3. Calcular probabilidades
        frecuencia, prob_total = tabla_probabilidades(frecuencia, total_bloques)

        # 4. Calcular entropía
        H = calcular_entropia(frecuencia)
        H_por_caracter = H / k


        # 5. Generar códigos Huffman
        codigos = huffman_mejorado(frecuencia)
        if not codigos:
            print("✗ Error: No se pudieron generar códigos Huffman")
            return None

        # 6. Calcular métricas
        L, L_real, eff, eff_real, tamaño_diccionario = calcular_metricas(
            frecuencia, H, total_bloques, codigos
        )

        # 7. Comprimir archivo - usar el texto COMPLETO, no la muestra
        nombre_base = nombre_archivo.split('.')[0]
        nombre_comprimido, tamaño_comprimido, overhead = comprimir_archivo(
            texto, frecuencia, codigos, nombre_base, k  # ← Usar 'texto' no 'texto_muestra'
        )
        if not nombre_comprimido:
            return None

        # 8. Validar descompresión
        valido = descomprimir_y_validar(nombre_comprimido, texto)  # ← Pasar texto completo

        # 9. Calcular relación de compresión
        tamaño_original = len(texto_original.encode('utf-8'))
        relacion_compresion = Decimal(tamaño_original) / tamaño_comprimido if tamaño_comprimido > 0 else Decimal(0)

        # 10. Guardar resultados
        resultado = {
        'archivo': nombre_archivo,
        'k': k,
        'simbolos': len(frecuencia),
        'entropia': float(H),
        'entropia_por_caracter': float(H_por_caracter),  # ✅ <--- AGREGAR ESTA LÍNEA
        'L_cx': float(L),
        'eficiencia': float(eff * 100),
        'tamaño_diccionario': tamaño_diccionario,
        'tamaño_comprimido': tamaño_comprimido,
        'overhead': overhead,
        'relacion_compresion': float(relacion_compresion),
        'valido': valido,
        'tiempo_procesamiento': time.time() - start_time,
        'tamaño_original': tamaño_original
        }

        # 11. Mostrar resultados con formato mejorado
        tabla_detalle = [
            ["Símbolos únicos", f"{len(frecuencia):,}"],
            ["Entropía H(X)", f"{H:.6f} bits/símbolo"],
            ["Entropía por carácter H(X)/k", f"{H_por_caracter:.6f} bits/carácter"],
            ["Longitud promedio L(C,X)", f"{L:.6f} bits/símbolo"],
            ["Eficiencia", f"{eff * 100:.2f}%"],
            ["Tamaño diccionario", f"{tamaño_diccionario:,} bytes"],
            ["Tamaño original", f"{tamaño_original:,} bytes"],
            ["Tamaño comprimido", f"{tamaño_comprimido:,} bytes"],
            ["Overhead", f"{overhead:,} bytes"],
            ["Relación compresión", f"{relacion_compresion:.2f}x"],
            ["Validación", "✓ ÉXITO" if valido else "✗ FALLO"],
            ["Tiempo procesamiento", f"{time.time() - start_time:.2f} segundos"]
        ]

        print(tabulate(tabla_detalle, tablefmt="grid"))

        return resultado

    except Exception as e:
        print(f"✗ Error procesando {nombre_archivo} con k={k}: {e}")
        import traceback
        traceback.print_exc()
        return None


def procesar_zip_archivos():
    """Procesa compresión ZIP para todos los archivos"""
    global resultados_zip

    print(f"\n{'=' * 80}")
    print("PROCESANDO COMPRESIÓN ZIP")
    print(f"{'=' * 80}")

    for nombre_archivo in archivos_diccionario.keys():
        start_time = time.time()

        try:
            # Leer archivo
            texto = leer_archivo(nombre_archivo)
            if not texto:
                continue

            tamaño_original = len(texto.encode('utf-8'))

            # Comprimir con ZIP
            nombre_base = nombre_archivo.split('.')[0]
            nombre_zip, tamaño_zip = comprimir_con_zip(texto, nombre_archivo)

            # Calcular relación de compresión
            relacion_compresion = Decimal(tamaño_original) / tamaño_zip if tamaño_zip > 0 else Decimal(0)

            resultado_zip = {
                'archivo': nombre_archivo,
                'metodo': 'ZIP',
                'tamaño_original': tamaño_original,
                'tamaño_comprimido': tamaño_zip,
                'relacion_compresion': float(relacion_compresion),
                'tiempo_procesamiento': time.time() - start_time
            }

            resultados_zip.append(resultado_zip)

            tabla_detalle_zip = [
                ["Método", "ZIP"],
                ["Archivo", nombre_archivo],
                ["Tamaño original", f"{tamaño_original:,} bytes"],
                ["Tamaño comprimido", f"{tamaño_zip:,} bytes"],
                ["Relación compresión", f"{relacion_compresion:.2f}x"],
                ["Tiempo procesamiento", f"{time.time() - start_time:.2f} segundos"]
            ]

            print(tabulate(tabla_detalle_zip, tablefmt="grid"))
            print()

        except Exception as e:
            print(f"✗ Error procesando ZIP para {nombre_archivo}: {e}")


################################################################################################
# 4. Ejecución principal
################################################################################################
def main():
    """Función principal que procesa todos los archivos y valores de k"""
    global resultados, resultados_zip

    print("INICIANDO PROCESAMIENTO DE ARCHIVOS...")
    print(f"Archivos a procesar: {list(archivos_diccionario.keys())}")
    print(f"Valores de k: {k_lista}")
    print()

    # Procesar compresión ZIP primero
    procesar_zip_archivos()

    # Procesar compresión Huffman para todos los k
    for nombre_archivo in archivos_diccionario.keys():
        for k in k_lista:
            resultado = procesar_archivo_completo(nombre_archivo, k)
            if resultado:
                resultados.append(resultado)

    # Mostrar resumen final con tabulate
    if resultados:
        print(f"\n{'#' * 120}")
        print("RESUMEN FINAL DE RESULTADOS HUFFMAN")
        print(f"{'#' * 120}")

        headers = ['Archivo', 'K', 'Símbolos', 'Entropía/token', 'Entropía/carácter', 'L(C,X)', 'Eficiencia%',
                   'Diccionario', 'Comprimido', 'Relación', 'Válido', 'Tiempo(s)']




        tabla_resumen = []
        for res in resultados:
            tabla_resumen.append([
                res['archivo'],
                res['k'],
                f"{res['simbolos']:,}",
                f"{res['entropia']:.4f}",            # entropía por token
                f"{res['entropia_por_caracter']:.4f}",  # entropía por carácter
                f"{res['L_cx']:.4f}",
                f"{res['eficiencia']:.2f}",
                f"{res['tamaño_diccionario']:,}",
                f"{res['tamaño_comprimido']:,}",
                f"{res['relacion_compresion']:.2f}x",
                '✓' if res['valido'] else '✗',
                f"{res['tiempo_procesamiento']:.2f}"
            ])

        # Imprimir tabla con formato elegante
        print(tabulate(tabla_resumen, headers=headers, tablefmt="grid", numalign="right", stralign="center"))

        # Mostrar comparación con ZIP
        print(f"\n{'#' * 120}")
        print("COMPARACIÓN HUFFMAN vs ZIP")
        print(f"{'#' * 120}")

        comparacion_headers = ['Archivo', 'Método', 'K', 'Tamaño', 'Relación', 'Tiempo(s)']
        comparacion_tabla = []

        # Agregar resultados ZIP
        for zip_res in resultados_zip:
            comparacion_tabla.append([
                zip_res['archivo'],
                'ZIP',
                '-',
                f"{zip_res['tamaño_comprimido']:,}",
                f"{zip_res['relacion_compresion']:.2f}x",
                f"{zip_res['tiempo_procesamiento']:.2f}"
            ])

        # Agregar resultados Huffman
        for huff_res in resultados:
            comparacion_tabla.append([
                huff_res['archivo'],
                'HUFFMAN',
                huff_res['k'],
                f"{huff_res['tamaño_comprimido']:,}",
                f"{huff_res['relacion_compresion']:.2f}x",
                f"{huff_res['tiempo_procesamiento']:.2f}"
            ])

        print(tabulate(comparacion_tabla, headers=comparacion_headers, tablefmt="grid", numalign="right",
                       stralign="center"))

        # Encontrar la mejor compresión por archivo
        print(f"\n{'#' * 120}")
        print("MEJOR MÉTODO DE COMPRESIÓN POR ARCHIVO")
        print(f"{'#' * 120}")

        mejores_headers = ['Archivo', 'Mejor Método', 'Mejor K', 'Mejor Relación', 'Tamaño']
        mejores_tabla = []

        for archivo in archivos_diccionario.keys():
            # Encontrar mejor resultado Huffman para este archivo
            huffman_results = [r for r in resultados if r['archivo'] == archivo]
            best_huffman = max(huffman_results, key=lambda x: x['relacion_compresion']) if huffman_results else None

            # Encontrar resultado ZIP para este archivo
            zip_result = next((r for r in resultados_zip if r['archivo'] == archivo), None)

            if best_huffman and zip_result:
                if best_huffman['relacion_compresion'] > zip_result['relacion_compresion']:
                    mejores_tabla.append([
                        archivo,
                        'HUFFMAN',
                        best_huffman['k'],
                        f"{best_huffman['relacion_compresion']:.2f}x",
                        f"{best_huffman['tamaño_comprimido']:,}"
                    ])
                else:
                    mejores_tabla.append([
                        archivo,
                        'ZIP',
                        '-',
                        f"{zip_result['relacion_compresion']:.2f}x",
                        f"{zip_result['tamaño_comprimido']:,}"
                    ])

        print(tabulate(mejores_tabla, headers=mejores_headers, tablefmt="grid", numalign="right", stralign="center"))

        # Guardar resultados en CSV
        with open('resultados_compresion.csv', 'w', encoding='utf-8') as f:
            # Escribir headers
            f.write(
                "Tipo;Archivo;K;Simbolos;Entropia;L_CX;Eficiencia%;Tamaño_Diccionario;Tamaño_Comprimido;Relacion_Compresion;Valido;Tiempo\n")

            # Escribir resultados Huffman
            for res in resultados:
                f.write(f"HUFFMAN;{res['archivo']};{res['k']};{res['simbolos']};{res['entropia']:.6f};")
                f.write(f"{res['L_cx']:.6f};{res['eficiencia']:.6f};{res['tamaño_diccionario']};")
                f.write(f"{res['tamaño_comprimido']};{res['relacion_compresion']:.6f};")
                f.write(f"{'SI' if res['valido'] else 'NO'};{res['tiempo_procesamiento']:.2f}\n")

            # Escribir resultados ZIP
            for res in resultados_zip:
                f.write(
                    f"ZIP;{res['archivo']};;;;;;;{res['tamaño_comprimido']};{res['relacion_compresion']:.6f};;{res['tiempo_procesamiento']:.2f}\n")

        print(f"\n✓ Resultados guardados en 'resultados_compresion.csv'")
        print(f"✓ Total de procesamientos Huffman: {len(resultados)}")
        print(f"✓ Total de procesamientos ZIP: {len(resultados_zip)}")
        print(f"✓ Procesamientos exitosos: {sum(1 for r in resultados if r['valido'])}")
    else:
        print("✗ No se completó ningún procesamiento exitoso")


if __name__ == "__main__":
    main()

INICIANDO PROCESAMIENTO DE ARCHIVOS...
Archivos a procesar: ['Quijote.txt', 'Unijote.txt']
Valores de k: [1, 2, 3, 4]


PROCESANDO COMPRESIÓN ZIP
+----------------------+-----------------+
| Método               | ZIP             |
+----------------------+-----------------+
| Archivo              | Quijote.txt     |
+----------------------+-----------------+
| Tamaño original      | 2,141,519 bytes |
+----------------------+-----------------+
| Tamaño comprimido    | 799,729 bytes   |
+----------------------+-----------------+
| Relación compresión  | 2.68x           |
+----------------------+-----------------+
| Tiempo procesamiento | 1.53 segundos   |
+----------------------+-----------------+

+----------------------+-----------------+
| Método               | ZIP             |
+----------------------+-----------------+
| Archivo              | Unijote.txt     |
+----------------------+-----------------+
| Tamaño original      | 2,559,040 bytes |
+----------------------+------------

**Discuta aquí los resultados ofreciendo una explicación para los mismos**

## **Resultados**

### **Diferencias entre los archivos: contenido y estructura**

#### `Quijote.txt` (texto natural):

* Es un texto literario con distribución **no uniforme** de caracteres
* Contiene **estructuras lingüísticas repetitivas**, patrones sintácticos y semánticos
* Tiene **alta redundancia**, lo que favorece la compresión

#### `Unijote.txt` (texto artificial/uniforme):

* Aparentemente un archivo con distribución de caracteres más aleatorizada
* **Menor redundancia** y menos patrones predecibles
* Dificulta la tarea de compresión basada en frecuencia, como Huffman.

### **Compresor ZIP vs. Huffman**

#### ZIP:

* Usa un enfoque combinado: diccionario (LZ77) + codificación entropía (Huffman estático)
* Detecta y reutiliza **repeticiones de patrones largos**, lo que lo hace muy eficaz en textos naturales
* Bajo costo computacional y buen equilibrio entre compresión y velocidad.

#### Huffman:

* Solo considera **frecuencia de símbolos**
* Más sensible a la calidad de la distribución de los datos de entrada.
* No reutiliza substrings ni aprovecha repetición de patrones largos

**Conclusión**: ZIP supera a Huffman en textos naturales como `Quijote.txt`


### **Impacto del valor de k en la compresión Huffman**

El valor de `k` indica cuántos caracteres se agrupan como símbolo para calcular la codificación de Huffman (k-gramas). Se esperaría que agrupar más caracteres ayude a capturar mejor los patrones y por tanto aumentar la compresión, pero:

#### Lo que sucede:

* Al aumentar `k`, el número de símbolos únicos crece:

  * `k=1`: pocos símbolos (\~90)
  * `k=2`: miles
  * `k=3`: cientos de miles
  * `k=4`: millones (en `Unijote.txt`)

* Esto causa:

  * Diccionarios gigantes (hasta 20 MB)
  * **Overhead** enorme que invalida cualquier ganancia por compresión
  * Archivos comprimidos más grandes que los originales
  * **Degradación de rendimiento** y aumento del tiempo de procesamiento

#### Entropía vs. Realidad:

* La **eficiencia teórica** sigue alta (\~99%), lo que indica que los códigos están bien ajustados a las frecuencias
* **En la práctica no compensa**, porque los nuevos símbolos no se repiten lo suficiente como para merecer codificaciones cortas.

**Conclusión**: Aunque la entropía por carácter disminuye con k (lo esperado), el crecimiento del espacio de simbolos y del diccionario anula los beneficios. Huffman no escala bien con k grande


### **Resultados por archivo**

#### En `Quijote.txt`:

* ZIP obtiene la **mejor compresión** (2.68x) con el menor tiempo (1.36s).
* Huffman solo es competitivo con **k=1** (1.85x); con k>1, el rendimiento cae
* A partir de k=3, la compresión se invierte: el archivo comprimido es **más grande** que el original

#### En `Unijote.txt`:

* Sorprendentemente, Huffman **con k=1** supera a ZIP (1.48x vs 1.30x)
* Esto puede deberse a una distribución más aprovechable por Huffman.
* A partir de k=2, la compresión se degrada rápidamente: diccionarios gigantes, compresión negativa y tiempos de procesamiento excesivos (hasta 217s en k=4)

**Conclusión**: En textos uniformes, Huffman con k pequeño puede ser más eficaz que ZIP, pero solo hasta cierto punto.


## **Explicación del fenómeno**

### ¿Por qué Huffman falla con k alto?

* **El algoritmo de Huffman** depende de **frecuencias relativas**. Si los símbolos son muy variados (k-gramas con alta cardinalidad), muchos tendrán frecuencias similares o bajas
* Esto significa que:

  * **Los códigos serán largos para la mayoría de los símbolos**, perdiendo la ventaja de asignar códigos cortos a los frecuentes
  * El diccionario que guarda la codificación **consume más espacio que lo que se ahorra al comprimir**
  * Aumenta el **overhead computacional** por tener que gestionar árboles más grandes y tablas de codificación más extensas